# IMPORT API KEY

In [1]:
import getpass
import os

if not os.environ.get("TAVILY_API_KEY"):
    os.environ["TAVILY_API_KEY"] = getpass.getpass("Tavily API key:\n")

# SEARCH TOOL DEFINITION

In [2]:
from typing import List, Literal, Optional
from langchain.tools import tool
from langchain_tavily import TavilySearch

@tool
def tavily_search(
    query: str,
    max_results: int = 5,
    topic: Literal["general", "news", "finance"] = "general",
    include_answer: bool = False,
    include_raw_content: bool = False,
    include_images: bool = False,
    include_image_descriptions: bool = False,
    search_depth: Literal["basic", "advanced"] = "basic",
    time_range: str = "",            # instead of None
    include_domains: List[str] = [], # instead of None
    exclude_domains: List[str] = [], # instead of None
    country: str = ""                # instead of None
) -> str:
    """
    Perform a web search using TavilySearch.

    Args:
        query: The search query string.
        max_results: Maximum number of search results to return.
        topic: Category of the search ("general", "news", "finance").
        include_answer: Include an answer to the original query in results.
        include_raw_content: Include cleaned and parsed HTML of each search result.
        include_images: Include a list of query-related images in the response.
        include_image_descriptions: Include descriptive text for each image.
        search_depth: Depth of the search ("basic" or "advanced").
        time_range: Time range to filter results ("day", "week", "month", "year") or "".
        include_domains: List of domains to specifically include.
        exclude_domains: List of domains to specifically exclude.
        country: Country to focus the search on.

    Returns:
        Search results as a string.
    """
    # parameters validation
    if len(time_range) == 0:
        time_range = None
    if len(include_domains) == 0:
        include_domains = None
    if len(exclude_domains) == 0:
        exclude_domains = None
    if len(country) == 0:
        country = None

    tool_instance = TavilySearch(
        max_results=max_results,
        topic=topic,
        include_answer=include_answer,
        include_raw_content=include_raw_content,
        include_images=include_images,
        include_image_descriptions=include_image_descriptions,
        search_depth=search_depth,
        time_range=time_range,
        include_domains=include_domains,
        exclude_domains=exclude_domains,
        country=country
    )

    return tool_instance.invoke({"query": query})

# EXTRACT TOOL DEFINITION

In [3]:
from typing import Literal
from langchain.tools import tool
from langchain_tavily import TavilyExtract

@tool
def tavily_extract(
    url: str,
    extract_depth: Literal["basic", "advanced"] = "basic",
    chunks_per_source: Literal[1, 2, 3, 4, 5] = 3,
    include_images: bool = False,
    include_favicon: bool = False,
    format_text: Literal["markdown", "text"] = "markdown"
) -> str:
    """
    Extract content from a web page using TavilyExtract.

    Args:
        url: The URL of the page to extract content from.
        extract_depth: Extraction depth ("basic" or "advanced").
        chunks_per_source: Number of chunks per source (1-5).
        include_images: Whether to include images (True or False).
        include_favicon: Whether to include favicon (True or False).
        format_text: Output format ("markdown" or "text").

    Returns:
        Extracted content as a string.
    """

    tool_instance = TavilyExtract(
        extract_depth=extract_depth,
        chunks_per_source=chunks_per_source,
        include_images=include_images,
        include_favicon=include_favicon,
        format=format_text,
    )

    return tool_instance.invoke({"urls": [url]})

# DYNAMIC TOOLS DEFINITION

In [4]:
from dataclasses import dataclass

@dataclass
class ToolType:
    tool_type: str = "search"

In [5]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing import Callable

@wrap_model_call
def dynamic_tool_call(request: ModelRequest, 
handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:

    """Dynamically call tools based on the runtime context"""

    tool_type = request.runtime.context.tool_type
    
    if tool_type == "search":
        tools = [tavily_search]
        request = request.override(tools=tools) 
    else:
        tools = [tavily_extract]
        request = request.override(tools=tools) 

    return handler(request)

# AGENT DEFINITION

In [6]:
from langchain.agents.structured_output import ToolStrategy
from langchain.agents import create_agent
from langchain_ollama import ChatOllama

model = ChatOllama(model="qwen3.5:9b")

agent = create_agent(
    model=model,
    tools=[tavily_search, tavily_extract],
    middleware=[dynamic_tool_call],
    context_schema=ToolType,
    # system_prompt = """
    #     You are a helpful assistant. You can execute the 'tavily_search' tool to perform a web search. 
        
    #     - The user will always provide a dictionary of parameters for the tool.
    #     - Only use the parameters provided by the user.
    #     - Execute the tool using these parameters.
    #     - The user's input will be in two parts: 'text' (query) and 'params' (dictionary of tool parameters).
    #     """,
)

# WEB SEARCH INVOCATION

In [7]:
import json

default_params = {    
    'max_results': 5,
    'topic': "general",
    'include_answer': False,
    'include_raw_content': False,
    'include_images': False,
    'include_image_descriptions': False,
    'search_depth': "basic",
    'time_range': "",
    'include_domains': [],
    'exclude_domains': [],
    'country': "",
}
user_params = {"include_domains": ["*.it"], "exclude_domains": ["*.com", "wikipedia.org"], "country": "italy"}
params = {**default_params, **user_params}
params_json = json.dumps(params)

user_query = "\"Pasta alla carbonara\""
query = f"text: {user_query}, params: {params_json}"

result = agent.invoke(
    {"messages": [{"role": "user", "content": query}]},
    context={"tool_type": "search"}
)

###### NOT STRUCTURED RESULT

In [8]:
result

{'messages': [HumanMessage(content='text: "Pasta alla carbonara", params: {"max_results": 5, "topic": "general", "include_answer": false, "include_raw_content": false, "include_images": false, "include_image_descriptions": false, "search_depth": "basic", "time_range": "", "include_domains": ["*.it"], "exclude_domains": ["*.com", "wikipedia.org"], "country": "italy"}', additional_kwargs={}, response_metadata={}, id='980c3785-bd10-4271-a96c-e3ef1d7d2bf4'),
  AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3.5:9b', 'created_at': '2026-04-07T08:22:11.3869986Z', 'done': True, 'done_reason': 'stop', 'total_duration': 104553720800, 'load_duration': 10104132400, 'prompt_eval_count': 605, 'prompt_eval_duration': 21425354800, 'eval_count': 263, 'eval_duration': 72765199400, 'model_name': 'qwen3.5:9b', 'model_provider': 'ollama'}, id='lc_run--019d6707-5442-7cb0-950b-28fe1d3dfa9c-0', tool_calls=[{'name': 'tavily_search_no_structured', 'args': {'query': 'Pasta alla carb

### Results filtering

In [9]:
import json

#sostituisci il rigo successivo con questa definizione
tool_msg = json.loads(result["messages"][-2].content)

research_results = tool_msg["results"]
research_results

[{'url': 'https://www.romahotelcressy.it/la-pasta-alla-carbonara/',
  'title': 'CARBONARA PASTA - Roma Hotel Cressy',
  'content': 'La pasta alla carbonara è un piatto tipico della cucina romana, ricco di storia e di varianti che si sono sviluppate nel corso del tempo. La sua origine esatta è ancora oggetto di dibattito, ma diverse teorie si concentrano sull’influenza delle razioni militari portate dagli eserciti alleati durante la Seconda Guerra Mondiale, che comprendevano uova e bacon, combinati con la pasta italiana. La carbonara ha anche suscitato interesse nella cultura popolare, con menzioni in canzoni e film, e addirittura con la celebrazione del “Carbonara Day” il 6 aprile di ogni anno, un evento sociale che celebra questa iconica ricetta italiana. La base della carbonara rimane comunque costituita da uova, pancetta o guanciale e formaggio pecorino. Aggiungere anche un po’ di acqua di cottura della pasta non salata per rendere la crema più morbida. Togliere il guanciale dalla p

# WEB PAGE EXTRACTION INVOCATION

In [12]:
from langchain.messages import HumanMessage

query = HumanMessage(content=[
    {"type": "text", "text": research_results[0]["url"]},
])

response = agent.invoke(
    {"messages": [query]},
    context={"tool_type": "extract"}
)

In [13]:
# DISPLAY TOOL MESSAGE
from IPython.display import Markdown, display

display(Markdown(response["messages"][-2].content))

{"results": [{"url": "https://www.romahotelcressy.it/la-pasta-alla-carbonara/", "title": "CARBONARA PASTA - Roma Hotel Cressy", "raw_content": "![it](https://www.romahotelcressy.it/wp-content/plugins/sitepress-multilingual-cms/res/flags/it.png)\n![en](https://www.romahotelcressy.it/wp-content/plugins/sitepress-multilingual-cms/res/flags/en.png)\n![Roma Hotel Cressy](https://www.romahotelcressy.it/wp-content/uploads/2020/09/HC-h95.jpg)\n\n### Main menu\n\n### LA PASTA ALLA CARBONARA\n\n![](https://www.romahotelcressy.it/wp-content/uploads/2024/04/Espaguetis_carbonara.jpeg)\n\n# LA PASTA ALLA CARBONARA\n\nLA STORIA\n\nLa pasta alla carbonara è un piatto tipico della cucina romana, ricco di storia e di varianti che si sono sviluppate nel corso del tempo.  \nLa sua origine esatta è ancora oggetto di dibattito, ma diverse teorie si concentrano sull’influenza delle razioni militari portate dagli eserciti alleati durante la Seconda Guerra Mondiale, che comprendevano uova e bacon, combinati con la pasta italiana.  \nTra le varie ipotesi, vi è quella anglosassone che suggerisce che i soldati americani durante l’occupazione di Roma abbiano combinato gli ingredienti a loro disposizione, dando così origine alla ricetta. Altre teorie attribuiscono l’origine della carbonara alla cucina napoletana o all’Appennino abruzzese, con ingredienti come cacio e uova.  \nNel corso degli anni, la ricetta si è evoluta e sono emerse numerose varianti, sia in Italia che all’estero. Alcune di queste varianti includono l’uso di formaggi diversi, l’aggiunta di panna, cipolle, funghi o sostituzioni di ingredienti per adattarsi a particolari restrizioni dietetiche, come nel caso delle versioni kosher.  \nLa carbonara ha anche suscitato interesse nella cultura popolare, con menzioni in canzoni e film, e addirittura con la celebrazione del “Carbonara Day” il 6 aprile di ogni anno, un evento sociale che celebra questa iconica ricetta italiana.\n\n**INGREDIENTI**  \nLa base della carbonara rimane comunque costituita da uova, pancetta o guanciale e formaggio pecorino.\n\nPer 4 persone\n\n1.Spaghetti 320 g.  \n2.Tuorli 6  \n3.Pepe nero q.b.  \n4.Guanciale 150 g.  \n5.Pecorino romano 50 g.\n\n**PREPARAZIONE**  \nRimuovere la cotenna e tagliare il guanciale a listarelle spesse. Metterlo in una padella a fuoco dolce fino a quando il grasso diventa lucido.   \nLavorare i tuorli con il pecorino e il pepe nero fino a ottenere una crema. Aggiungere un po’ del grasso del guanciale e mescolare. Aggiungere anche un po’ di acqua di cottura della pasta non salata per rendere la crema più morbida.   \nCuocere la pasta in acqua non troppo salata fino a quando è al dente. Scolarla, mantenendo un po’ di acqua di cottura nella padella con il guanciale.   \nTogliere il guanciale dalla padella e aggiungere la pasta scolata. Unire la crema di tuorli e formaggio alla pasta e amalgamare bene a fuoco spento. Aggiungere il guanciale sopra la pasta una volta impiattata.   \nCompleta il piatto con pecorino e pepe nero macinato fresco.   \nSeguendo questi passaggi, otterrai una deliziosa pasta alla carbonara cremosa e piena di sapore. Buon appetito!\n\nCREDITS FOTO  \nJavier Somoza, [CC BY-SA 4.0, via Wikimedia Commons](https://creativecommons.org/licenses/by-sa/4.0)\n\nPrenota il tuo soggiorno a Roma in maniera diretta **risparmia sulle commissioni**\n\n## TEL: +39 06 48 49 17 hotelcressy@gmail.com\n\n[![](https://www.romahotelcressy.it/wp-content/uploads/2023/05/WA-150x150.jpg)](https://api.whatsapp.com/send?phone=393517634333&text=Salve%20Hotel%20Cressi%20Roma,%20vorrei%20chattare%20con%20voi)\n\n![](https://www.romahotelcressy.it/wp-content/uploads/2023/05/WA-150x150.jpg)\n\nComments are closed.\n\n### Search\n\n### Recent Posts\n\n![](https://www.romahotelcressy.it/wp-content/uploads/2015/06/TripAdvisor-logo-300x51.png)", "images": []}], "failed_results": [], "response_time": 0.01, "request_id": "ac3cbcfc-4cd0-45c9-9870-796891ad3963"}

In [14]:
# DISPLAY AI MESSAGE
from IPython.display import Markdown, display

display(Markdown(response["messages"][-1].content))

# 🍝 Pasta alla Carbonara - Roma Hotel Cressy

Ecco i dettagli completi della ricetta della pasta alla carbonara dal sito del Roma Hotel Cressy:

---

## 📖 **LA STORIA**

La pasta alla carbonara è un piatto tipico della cucina romana, ricco di storia e di varianti. L'origine esatta è ancora dibattuta, ma le teorie principali includono:

- **Teoria militare**: razioni degli eserciti alleati durante la Seconda Guerra Mondiale (uova + bacon)
- **Influenza anglosassone**: soldati americani che combinasero gli ingredienti a disposizione
- **Origini napoletane**: cucina napoletana con cacio e uova
- **Origini abruzzesi**: varianti con cacio e uova dall'Appennino

Nel corso degli anni sono emerse molte varianti internazionali con formaggi diversi, panna, cipolle, funghi, o adattamenti per restrizioni dietetiche.

---

## 🥘 **INGREDIENTI** (Per 4 persone)

| Quantità | Ingrediente |
|----------|-------------|
| 320 g | Spaghetti |
| 6 | Tuorli |
| Q.b. | Pepe nero |
| 150 g | Guanciale |
| 50 g | Pecorino romano |

---

## 👨‍🍳 **PREPARAZIONE**

1. **Preparare il guanciale**: Rimuovere la cotenna, tagliare a listarelle spesse, cuocere in padella a fuoco dolce fino a quando il grasso diventa lucido

2. **Preparare la crema**: Lavorare i tuorli con il pecorino e il pepe nero fino a ottenere una crema, aggiungendo un po' del grasso del guanciale e un po' di acqua di cottura

3. **Cuocere la pasta**: Cospargere la pasta in acqua non troppo salata fino a quando è al dente

4. **Completare**: Scolarla, mantenere la padella con il guanciale caldo

5. **Assemblare**: Togliere il guanciale dalla padella, aggiungere la pasta, unire la crema di tuorli e formaggio amalgamando a fuoco spento

6. **Servire**: Aggiungere il guanciale sopra la pasta una volta impiattata, completare con pecorino e pepe nero macinato fresco

---

## 📞 **Contatti Roma Hotel Cressy**

- **Telefono**: +39 06 48 49 17
- **Email**: hotelcressy@gmail.com
- **Risparmia**: prenota direttamente per evitare commissioni

---

💡 **Consiglio**: La carbonara tradizionale NON contiene panna o cipolla! È solo uova, guanciale, pecorino, pepe e pasta.